# Suite JSN — JSON

JSON is a thin, strict (RFC 8259) encoding of Plain data. These cases cover strictness on output and input, text
forms and encodings, value fidelity, and agreement with Plain.

In [ ]:
import json
import math

from mbse.Schemas.Framework import JSON, Plain, Proxies, Schemas as S
from mbse.Schemas.Framework.Errors import DecodeError
from support import raises, same_graph, text

store = Proxies.OfStore()

Item = S.OfObject.Builder().name("Item").ref().properties(text("s"), text("i", int), text("f", float), text("b", bool),
                                       text("raw", bytes)).create()
Link = S.OfRelation.Builder().name("Link").links("from", "to").properties(text("w", float)).create()
Item = S.OfObject.Builder(Item).relations(lambda r: r.name("out").of(Link).me("from"),
                                          lambda r: r.name("in").of(Link).me("to")).update()
store.register(Item)
store.register(Link)
B = store
FromJSON = JSON.FromJSON(B)

## JSN-01 · Output never contains NaN or Infinity literals

In [ ]:
for value in [math.nan, math.inf, -math.inf]:
    out = JSON.ToJSON(store)(Item, B.Item().f(value).create())
    json.loads(out, parse_constant=lambda c: (_ for _ in ()).throw(AssertionError(c)))
    assert json.loads(out)["objects"]["s0"]["f"] in ("NaN", "Infinity", "-Infinity")
with raises(ValueError):
    JSON.dumps({"x": math.nan})  # plain data should never hold one, and dumps refuses it

## JSN-02 · Input: non-standard constants, duplicate keys and syntax errors are rejected

In [ ]:
for bad in ['{"a": NaN}', '{"a": Infinity}', '{"a": -Infinity}', '[NaN]',
            '{"a": 1, "a": 2}', '{"x": {"a": 1, "a": 1}}', '[{"k": 1, "k": 1}]',
            '{"a": 1,}', "{'a': 1}", '{"a": 1} // c', '{"a": 01}', '', '   ', '{"a": 1} {"b": 2}', '{"a": .5}']:
    with raises(DecodeError, match="line 1, column "):
        JSON.loads(bad)

## JSN-03 · Text forms: str, UTF-8/16/32 bytes, and byte-order marks

In [ ]:
doc = '{"a": "é"}'
assert JSON.loads(doc) == JSON.loads(doc.encode("utf-8")) == {"a": "é"}
for encoding in ["utf-16", "utf-16-le", "utf-16-be", "utf-32", "utf-8-sig"]:
    assert JSON.loads(doc.encode(encoding)) == {"a": "é"}, encoding
with raises(DecodeError, match="line 1, column 1: unexpected byte order mark"):
    JSON.loads("﻿" + doc)  # a BOM inside a str is an error; in bytes it is accepted
with raises(DecodeError, match="input is not valid utf-16"):
    JSON.loads(b"\xff\xfe\x00")  # undecodable bytes

## JSN-04 · Output format: key order kept, non-ASCII kept, indentation optional

In [ ]:
item = B.Item().raw(b"\x01").s("日本").i(1).create()
compact = JSON.ToJSON(store)(Item, item)
assert compact == '{"root": "s0", "objects": {"s0": {"s": "日本", "i": 1, "raw": "AQ=="}}}'
pretty = JSON.ToJSON(store)(Item, item, indent=2)
assert "\n" in pretty and JSON.loads(pretty) == JSON.loads(compact)
assert JSON.ToJSON(store).OfObject(Item, item, indent=4) == JSON.dumps(Plain.ToPlain(store)(Item, item), indent=4)

## JSN-05 · Value fidelity: integers of any size, exact floats, every string

In [ ]:
for value in [0, -1, 2**53 + 1, 2**100, -(2**200)]:
    assert FromJSON(Item, JSON.ToJSON(store)(Item, B.Item().i(value).create())).i == value
for value in [0.1, 1 / 3, 1e308, 5e-324, 2.2250738585072014e-308, -0.0, 123456789.123456789]:
    back = FromJSON(Item, JSON.ToJSON(store)(Item, B.Item().f(value).create())).f
    assert back == value and math.copysign(1, back) == math.copysign(1, value)
for value in ["", "\x00", "\x1f\x7f", '"\\/', "  ", "\ud800", "\U0001f600", "a\r\nb"]:
    assert FromJSON(Item, JSON.ToJSON(store)(Item, B.Item().s(value).create())).s == value, repr(value)
assert FromJSON(Item, JSON.ToJSON(store)(Item, B.Item().raw(bytes(range(256))).create())).raw == bytes(range(256))

## JSN-06 · A lone surrogate is valid Python JSON text but cannot be encoded to UTF-8 (pinned)

In [ ]:
lone = JSON.ToJSON(store)(Item, B.Item().s("\ud800").create())
with raises(UnicodeEncodeError):
    lone.encode("utf-8")
assert "\ud800" in lone  # callers writing files should use errors="surrogatepass" or reject such strings

## JSN-07 · Natives at the top level

In [ ]:
assert JSON.ToJSON(store)(S.OfNative.Data(int), 5) == "5"
assert JSON.ToJSON(store)(S.OfNative.Data(bytes), b"\xff") == '"/w=="'
assert JSON.ToJSON(store).OfNative(S.OfNative.Data(float), math.inf) == '"Infinity"'
assert FromJSON(S.OfNative.Data(str), '"x"') == FromJSON.OfNative(S.OfNative.Data(str), '"x"') == "x"
with raises(DecodeError, match="$: expected int, got bool"):
    FromJSON(S.OfNative.Data(int), "true")
with raises(DecodeError, match="$: expected float, got int"):
    FromJSON(S.OfNative.Data(float), "1")  # JSON 1 is an integer, not a float

## JSN-08 · JSON agrees with Plain, and graphs round trip

In [ ]:
hub = B.Item().s("hub").out(lambda x: x.to(lambda y: y.s("leaf1")).w(1.0)).out(
    lambda x: x.to(lambda y: y.s("leaf2")).w(math.nan)).create()
graph = Plain.ToPlain(store).Reachable(Item, hub)
text_ = JSON.ToJSON(store).Reachable(Item, hub)
assert JSON.loads(text_) == graph
back = FromJSON.Reachable(Item, text_)
assert back is not hub and same_graph(Plain.ToPlain(store).Reachable(Item, back), graph)
with raises(DecodeError, match="$.objects.s0.out[0].to: unresolved reference 's1'"):
    FromJSON(Item, JSON.ToJSON(store)(Item, hub))
with raises(DecodeError, match="unresolved reference 's9'"):
    FromJSON.Reachable(Item, text_.replace('"s1"', '"s9"', 1))

## JSN-09 · Every input problem is a `DecodeError` with the same reason and position in every binding
The table is shared verbatim with the TypeScript suite. Rows: input (text, or bytes as a list), the expected message
(or None when it parses), and the repr of the parsed value. Columns count code points; undecodable bytes have no
location.

In [ ]:
import json as stdlib_json

TABLE = stdlib_json.loads(r'''[["", "line 1, column 1: expecting a value", null], ["   ", "line 1, column 4: expecting a value", null], ["-", "line 1, column 1: expecting a value", null], ["nul", "line 1, column 1: expecting a value", null], ["tru", "line 1, column 1: expecting a value", null], [".5", "line 1, column 1: expecting a value", null], ["[1,]", "line 1, column 3: trailing comma before ']'", null], ["{\"a\": 1,}", "line 1, column 8: trailing comma before '}'", null], ["{\"a\" 1}", "line 1, column 6: expecting ':'", null], ["{1: 2}", "line 1, column 2: expecting a property name in double quotes", null], ["{\"a\": 1 \"b\": 2}", "line 1, column 9: expecting ',' or '}'", null], ["[1 2]", "line 1, column 4: expecting ',' or ']'", null], ["\"abc", "line 1, column 1: unterminated string", null], ["\"a\\", "line 1, column 1: unterminated string", null], ["\"a\\x\"", "line 1, column 3: invalid escape", null], ["\"\\u12\"", "line 1, column 3: invalid \\uXXXX escape", null], ["\"\\uZZZZ\"", "line 1, column 3: invalid \\uXXXX escape", null], ["\"a\nb\"", "line 1, column 3: invalid control character in a string", null], ["\"a\tb\"", "line 1, column 3: invalid control character in a string", null], ["1 2", "line 1, column 3: unexpected data after the value", null], ["{\"a\": 1} x", "line 1, column 10: unexpected data after the value", null], ["[", "line 1, column 2: expecting a value", null], ["{", "line 1, column 2: expecting a property name in double quotes", null], ["{\"a\"", "line 1, column 5: expecting ':'", null], ["{\"a\":", "line 1, column 6: expecting a value", null], ["01", "line 1, column 2: unexpected data after the value", null], ["1.", "line 1, column 2: unexpected data after the value", null], ["1e", "line 1, column 2: unexpected data after the value", null], ["[\n  1,\n  x]", "line 3, column 3: expecting a value", null], ["{\n\"a\": [\n1,\n2\n]\n,}", "line 6, column 1: trailing comma before '}'", null], ["NaN", "line 1, column 1: NaN is not valid JSON", null], ["-Infinity", "line 1, column 1: -Infinity is not valid JSON", null], ["[Infinity]", "line 1, column 2: Infinity is not valid JSON", null], ["99999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999", "line 1, column 1: an integer has 4301 digits, more than the limit of 4300", null], [[0, 0, 0, 49], null, "1"], [[49, 0, 0, 0], null, "1"], [[254, 255, 0, 49], null, "1"], [[0, 49], null, "1"], [[49, 0], null, "1"], [[0, 0, 254, 255, 0, 0, 0, 49], null, "1"], [[0, 0, 0], "line 1, column 1: expecting a value", null], [[255, 254, 49], "input is not valid utf-16", null], [[195, 40], "input is not valid utf-8", null], [[255, 254, 0, 0, 0, 0, 17, 0], "input is not valid utf-32", null], [[0, 0, 0, 49, 50], "input is not valid utf-32-be", null], [[0, 0, 0, 49, 50, 51], "input is not valid utf-32-be", null], [[239, 187, 191, 49], null, "1"], ["\"🚀\\x\"", "line 1, column 3: invalid escape", null], ["{\"a\": 1, \"a\": 2}", "line 1, column 10: duplicate key 'a'", null], ["{\"a\": {\"b\": 1, \"b\": 2}}", "line 1, column 16: duplicate key 'b'", null], ["[1, NaN]", "line 1, column 5: NaN is not valid JSON", null], ["﻿1", "line 1, column 1: unexpected byte order mark", null], ["\n\n  [1,]", "line 3, column 5: trailing comma before ']'", null], ["{\"a\": 1}\r\n x", "line 2, column 2: unexpected data after the value", null], ["\"\\ud83d\"", null, "'\\ud83d'"], ["[1] [2]", "line 1, column 5: unexpected data after the value", null], [[239, 187, 191, 91, 93], null, "[]"], [[254, 255, 216, 0], "input is not valid utf-16", null], [[255], "input is not valid utf-8", null], ["[\"🚀🚀\", x]", "line 1, column 8: expecting a value", null], [[0, 0, 216, 0, 0, 0, 0, 49], "input is not valid utf-32-be", null], [[237, 160, 128], "input is not valid utf-8", null]]''')
assert len(TABLE) == 62
for source, message, result in TABLE:
    data = bytes(source) if isinstance(source, list) else source
    try:
        value = JSON.loads(data)
    except DecodeError as error:
        assert message is not None, (source, str(error))
        assert str(error) == message and str(error).count("\n") == 0, (source, str(error))
        assert (error.line is None) == message.startswith("input is not valid") and error.path is None
    else:
        assert message is None and repr(value) == result, (source, value)

# Python reads valid input with `json` and runs the reference parser only to report a problem, so the reference parser
# must read every valid input exactly as `json` does. (In the TypeScript binding the reference parser reads all input.)
VALID = ["null", "true", "false", "0", "-12", "1.5", "-0.0", "1e3", "2.5E-3", r'"a\"\\\/\b\f\n\r\t"',
         r'"\u00e9\ud83d\ude00\ud800\udc00x"', r'"\ud800\u0041"', "[]", '[1, [2, {}], {"a": [null]}]',
         '{"a": 1, "b": {"c": "d"}}', " \n\t\r{ } ", '"日本語 🚀"']
for text in VALID + [source for source, message, _ in TABLE if message is None and isinstance(source, str)]:
    assert repr(JSON._Parser(text).parse()) == repr(stdlib_json.loads(text)), text
assert JSON.loads('["\\ud83d\\ude00", "\\ud800", "\\ud800x", "\\udc00", "\\b\\f\\n\\r\\t\\/\\\\\\""]') == [
    "\U0001f600", "\ud800", "\ud800x", "\udc00", "\b\f\n\r\t/\\\""]

## JSN-10 · Only plain data can be written
Python's `json` would turn an int, float, bool or None key into a string; plain data only has string keys.

In [ ]:
with raises(TypeError, match="keys must be str, not int"):
    JSON.dumps({1: 2})
with raises(TypeError, match="keys must be str, not int"):
    JSON.dumps({"a": [{"b": {1: 2}}]})
for value, name in [({1, 2}, "set"), (b"x", "bytes"), (object(), "object")]:
    with raises(TypeError, match=f"Object of type {name} is not JSON serializable"):
        JSON.dumps({"a": [value]})

## JSN-11 · Every kind of plain value is written and read, in Python's exact text

In [ ]:
value = [None, True, False, 0, -1, 1.5, "s", [], {}, {"a": [None, {}]}, [[]]]
assert JSON.dumps(value) == '[null, true, false, 0, -1, 1.5, "s", [], {}, {"a": [null, {}]}, [[]]]'
assert JSON.dumps(value, indent=1) == '[\n null,\n true,\n false,\n 0,\n -1,\n 1.5,\n "s",\n [],\n {},\n {\n  "a": [\n   null,\n   {}\n  ]\n },\n [\n  []\n ]\n]'
assert JSON.loads(JSON.dumps(value)) == value and JSON.loads(JSON.dumps(value, indent=1)) == value
assert JSON.loads(" null ") is None and JSON.loads("{}") == {} and JSON.loads("[ ]") == [] and JSON.loads('{ }') == {}